# 05 -- HuggingFace name embeddings

Notebook 03's `name_uniqueness` feature captures *repetition* -- 1.0 if a
candidate's name doesn't repeat elsewhere in the table, lower the more it
does, 0.0 if unnamed. It captures nothing about what the name actually
*means*. This notebook tests whether embedding candidate names with a
pretrained HuggingFace sentence-transformer, and adding that as engineered
features, improves on Notebook 03's feature set.

**Stated honestly going in:** `rating` measures *visual* spottability from
the air. Name semantics are a weak, indirect signal for that at best -- a
lake being called "Long Lake" doesn't make it more or less visible.
"No improvement" is a legitimate, useful result here, not a failure of the
approach -- it says something real about which feature sources matter for
this task. This notebook runs inside the Docker container, same as
Notebook 04.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## Step 1 -- Load the same modeling table as Notebook 03

In [2]:
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features_c81_kdlh.parquet"
LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "spottability_ratings.csv"

from vfr import pipeline

# The labelled table and the split every trainer in this project uses --
# vfr.pipeline.labeled_split, the one retrain() and the candidate models
# call, so these numbers are comparable with the Dev console's. It reads
# the chart picks as well as the older ratings file, fills a missing
# name_uniqueness, and knows which feature columns there are.
split = pipeline.labeled_split(FEATURES_PATH, LABELS_PATH)
labeled_df, FEATURE_COLS = split.labeled, split.feature_cols
X_base = labeled_df[FEATURE_COLS]
y = labeled_df["rating"].astype(float)

print(f"{len(labeled_df)} labeled candidates, {len(FEATURE_COLS)} baseline features")

205 labeled candidates, 12 baseline features


## Step 2 -- Embed names

`all-MiniLM-L6-v2` -- small, fast, well-suited to short text like place
names. Unnamed candidates (143 of 230, see Notebook 03) don't get a
zero-vector -- that would be an arbitrary out-of-distribution point the
model has to special-case. Instead they get embedded as `"unnamed
<category>"` (e.g. `"unnamed lake or pond"`), so the embedding still
carries real semantic content, just about the category rather than a
specific name.

In [3]:
from sentence_transformers import SentenceTransformer

embed_text = labeled_df.apply(
    lambda r: r["name"] if isinstance(r["name"], str) and r["name"].strip()
    else f"unnamed {r['category'].replace('_', ' ')}",
    axis=1,
)

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(embed_text.tolist(), show_progress_bar=False)

print("embeddings shape:", embeddings.shape)
embed_text.head(10)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embeddings shape: (205, 384)


0             Superior
1             Montello
2           Round Lake
3     Round Lake Beach
4    Big Falls Flowage
5         Tagalder Bay
6            Wood Lake
7           Hahns Lake
8    Upper Spring Lake
9           Mills Lake
dtype: str

## Step 3 -- Reduce to a handful of components

384 embedding dimensions against 228 samples is a bad ratio -- feeding raw
embedding dims into the models would mostly add noise. PCA down to 5
components, fit on the labeled candidates' embeddings -- an unsupervised
transform of the *text*, no `rating` values involved, so it doesn't leak
target information the way fitting something on labels would.

In [4]:
N_COMPONENTS = 5
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
embed_pca = pca.fit_transform(embeddings)

print("explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("total explained:", round(pca.explained_variance_ratio_.sum(), 3))

embed_cols = [f"name_embed_pca_{i}" for i in range(N_COMPONENTS)]
embed_df = pd.DataFrame(embed_pca, columns=embed_cols, index=labeled_df.index)

X_augmented = pd.concat([X_base, embed_df], axis=1)
X_augmented.head()


explained variance ratio: [0.175 0.076 0.059 0.054 0.043]
total explained: 0.408


,log_size,elevation_prominence_m,name_uniqueness,nn_dist_nm,category_airport,category_intersection,category_lake_or_pond,category_railroad,category_river,category_stadium,category_town,category_wind_farm,name_embed_pca_0,name_embed_pca_1,name_embed_pca_2,name_embed_pca_3,name_embed_pca_4
0,0.0,2.880959,1.0,0.008846,False,False,False,False,False,False,True,False,0.107701,-0.313126,-0.142465,0.137457,-0.056004
1,0.0,-15.898533,1.0,0.062157,False,False,False,False,False,False,True,False,-0.109144,-0.191808,-0.098866,-0.083409,0.289337
2,0.0,6.072186,1.0,0.035830,False,False,False,False,False,False,True,False,-0.391436,0.213654,0.186057,0.197291,0.034213
3,0.0,-4.861698,1.0,0.731261,False,False,False,False,False,False,True,False,-0.305710,0.162215,0.186402,0.260026,0.060463
4,0.0,-9.940987,1.0,0.727215,False,False,True,False,False,False,False,False,-0.116378,-0.292535,-0.098148,-0.159928,-0.097680


## Step 4 -- Nested CV: baseline vs baseline + embeddings

Same nested-CV setup as Notebook 03's Step 12 -- an outer 5-fold loop for
an honest, un-tuned-on performance estimate, an inner 5-fold loop for
hyperparameter search -- run twice: once on `X_base`, once on
`X_augmented`. Same models, same grids, same folds. This directly answers
"do the embeddings help" without introducing a new evaluation scheme to
second-guess.

In [5]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "Ridge": (Ridge(random_state=RANDOM_STATE), {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "RandomForest": (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 3, 5]},
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [2, 3, 4], "learning_rate": [0.01, 0.05, 0.1]},
    ),
}

dummy_mae = -cross_val_score(DummyRegressor(strategy="mean"), X_base, y, cv=outer_cv, scoring="neg_mean_absolute_error")
print(f"{'Dummy (mean)':>17}: {dummy_mae.mean():.3f} +/- {dummy_mae.std():.3f}")

results = {}
for label, X_variant in [("baseline", X_base), ("baseline + embeddings", X_augmented)]:
    print(f"\n-- {label} --")
    for name, (estimator, grid) in param_grids.items():
        search = GridSearchCV(estimator, grid, cv=inner_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
        scores = -cross_val_score(search, X_variant, y, cv=outer_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
        results[(label, name)] = scores
        print(f"{name:>17}: {scores.mean():.3f} +/- {scores.std():.3f}")


     Dummy (mean): 1.143 +/- 0.075

-- baseline --


            Ridge: 1.076 +/- 0.073


     RandomForest: 0.955 +/- 0.083


 GradientBoosting: 0.980 +/- 0.068

-- baseline + embeddings --


            Ridge: 1.069 +/- 0.075


     RandomForest: 0.943 +/- 0.101


 GradientBoosting: 1.002 +/- 0.075


## Step 5 -- Verdict

A side-by-side table, plus the actual read: any differences here need to
be bigger than the fold-to-fold std to mean anything -- these are 5-number
estimates, not exact values.

In [6]:
summary = pd.DataFrame([
    {
        "model": name,
        "baseline_MAE": results[("baseline", name)].mean(),
        "baseline_std": results[("baseline", name)].std(),
        "with_embeddings_MAE": results[("baseline + embeddings", name)].mean(),
        "with_embeddings_std": results[("baseline + embeddings", name)].std(),
    }
    for name in param_grids
]).set_index("model")

summary["delta"] = summary["with_embeddings_MAE"] - summary["baseline_MAE"]
summary


,baseline_MAE,baseline_std,with_embeddings_MAE,with_embeddings_std,delta
model,,,,,
Ridge,1.076237,0.072990,1.069033,0.074986,-0.007203
RandomForest,0.955012,0.083494,0.942615,0.100840,-0.012397
GradientBoosting,0.980182,0.067939,1.002243,0.075156,0.022061


Every `delta` here is well inside the corresponding std band -- the
embeddings neither reliably help nor hurt. That's the expected, legitimate
result stated up front: name text just doesn't carry visual-spottability
signal for this task. Notebook 03's original feature set stays the one to
carry forward; this notebook's value is having actually tested the
alternative rather than assumed the answer.